# 01 Bronze ingest

Raw volume JSON → **schema-enforced** Delta tables. Bronze layer keeps the data as parsed
(no cleaning); it only enforces a schema and makes re-ingestion idempotent.

- **`bronze_battles`**: one row per `battle_id`, loaded with an idempotent `MERGE`.
  The same battle appears in two players' logs, so the `MERGE` collapses those to one row.
- **`bronze_cards`**: small, slowly-changing dim; full-refresh overwrite each run.
- **`bronze_players`**: the crawl seed, kept for lineage (overwrite).

## Config

In [0]:
from pyspark.sql import functions as F, Window, types as T
from delta.tables import DeltaTable

CATALOG, SCHEMA = "workspace", "clash"
RAW = f"/Volumes/{CATALOG}/{SCHEMA}/raw"

BRONZE_BATTLES = f"{CATALOG}.{SCHEMA}.bronze_battles"
BRONZE_CARDS   = f"{CATALOG}.{SCHEMA}.bronze_cards"
BRONZE_PLAYERS = f"{CATALOG}.{SCHEMA}.bronze_players"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

DataFrame[]

## Bronze battles
Schema enforcement, deduplication, and idempotent merge are applied in the bronze battlelog layer.

In [0]:
battles_schema = T.StructType([
    T.StructField("battle_id", T.StringType()),
    T.StructField("battle_time", T.StringType()),
    T.StructField("battle_time_raw", T.StringType()),
    T.StructField("type", T.StringType()),
    T.StructField("is_ladder", T.BooleanType()),
    T.StructField("game_mode", T.StringType()),
    T.StructField("arena_id", T.LongType()),
    T.StructField("arena_name", T.StringType()),
    T.StructField("player_tag", T.StringType()),
    T.StructField("player_name", T.StringType()),
    T.StructField("player_starting_trophies", T.LongType()),
    T.StructField("player_trophy_change", T.LongType()),
    T.StructField("player_crowns", T.LongType()),
    T.StructField("player_cards", T.ArrayType(T.LongType())),
    T.StructField("player_deck_hash", T.StringType()),
    T.StructField("opponent_tag", T.StringType()),
    T.StructField("opponent_name", T.StringType()),
    T.StructField("opponent_starting_trophies", T.LongType()),
    T.StructField("opponent_trophy_change", T.LongType()),
    T.StructField("opponent_crowns", T.LongType()),
    T.StructField("opponent_cards", T.ArrayType(T.LongType())),
    T.StructField("opponent_deck_hash", T.StringType()),
    T.StructField("win", T.BooleanType()),
    T.StructField("source_player_tag", T.StringType()),
    T.StructField("ingested_at", T.StringType()),
])

raw = spark.read.schema(battles_schema).json(f"{RAW}/*/battles.jsonl")

In [0]:
# Deduplicate within this load so each battle_id appears once before the MERGE.
# Always keep the row with the lowest source_player_tag for a battle, so the kept row is deterministic across reruns.
w = Window.partitionBy("battle_id").orderBy("source_player_tag")
staged = (raw
    .withColumn("rn", F.row_number().over(w))
    .filter("rn = 1").drop("rn")
    .withColumn("bronze_loaded_at", F.current_timestamp()))

print(raw.count(), "raw rows ->", staged.count(), "unique battle_id")

317467 raw rows -> 315798 unique battle_id


### Idempotent MERGE

First run creates the Delta table; later runs `MERGE` on `battle_id` and insert only battles not already there. This ensures re-running the pipeline does not create duplicates or overwrite existing battles.

In [0]:
if not spark.catalog.tableExists(BRONZE_BATTLES):
    staged.write.format("delta").saveAsTable(BRONZE_BATTLES)
    print("created", BRONZE_BATTLES)
else:
    (DeltaTable.forName(spark, BRONZE_BATTLES).alias("t")
        .merge(
            source=staged.alias("s"),
            condition="t.battle_id = s.battle_id"
        ).whenNotMatchedInsertAll()
        .execute())
    print("merged into", BRONZE_BATTLES)

merged into workspace.clash.bronze_battles


In [0]:
bb = spark.table(BRONZE_BATTLES)
print(bb.count(), "rows;", bb.select("battle_id").distinct().count(), "distinct battle_id")
display(bb.limit(10))

315798 rows; 315798 distinct battle_id


battle_id,battle_time,battle_time_raw,type,is_ladder,game_mode,arena_id,arena_name,player_tag,player_name,player_starting_trophies,player_trophy_change,player_crowns,player_cards,player_deck_hash,opponent_tag,opponent_name,opponent_starting_trophies,opponent_trophy_change,opponent_crowns,opponent_cards,opponent_deck_hash,win,source_player_tag,ingested_at,bronze_loaded_at
000018fc534796f8cb0aaa51b5a7b3f647912cec,2026-05-31T19:16:18+00:00,20260531T191618.000Z,pathOfLegend,false,Ranked1v1_NewArena2,54000145,Legendary Arena,#8JQRP88P2,@victorstruzani,null,30,1,"List(27000002, 26000034, 26000056, 28000015, 26000010, 26000053, 28000000, 26000005)",365903759de62a4cdf850c201a0aef98b6dacf76,#80Y2R9YLV,Ragnar Lothbrok,null,null,0,"List(26000064, 26000034, 26000027, 26000001, 26000016, 26000045, 28000015, 28000012)",60e8d6d402b63d4e37c1fd565231b5efd7e7b5d3,true,#8JQRP88P2,2026-06-02T16:28:26.289237+00:00,2026-06-03T16:04:50.797Z
0002220950bd20de93a497997619b823e94afc4c,2026-05-25T08:23:40+00:00,20260525T082340.000Z,riverRaceDuel,false,CW_Duel_1v1,54000144,Spirit Square,#L9CQ8GVC,Daniel Manduuu,14000,null,2,"List(26000050, 26000062, 26000059, 26000038, 27000001, 28000018, 28000007, 28000016, 26000064, 26000065, 27000006, 26000021, 28000011, 28000014, 26000030, 26000010, 26000024, 26000077, 26000044, 26000061, 28000000, 28000015, 26000084, 26000025)",947bc9e3195823a4f2991531601900f81d8f64ba,#L00Q0YVLL,Ex gamerB,13920,null,2,"List(26000059, 26000000, 26000047, 26000080, 27000012, 26000083, 26000084, 28000000, 26000055, 26000062, 26000064, 26000049, 28000013, 26000056, 26000020, 26000012, 26000036, 26000057, 26000004, 26000042, 28000009, 26000046, 28000008, 26000050)",a11dad373ea3a64ae6e20bf1b13d46a1f793681c,false,#L9CQ8GVC,2026-06-02T16:33:17.410682+00:00,2026-06-03T16:04:50.797Z
0002687d540b39ea371b6353e1b11826eecebaa1,2026-05-28T05:19:25+00:00,20260528T051925.000Z,pathOfLegend,false,Ranked1v1_NewArena2,54000145,Legendary Arena,#2899L9L,Hossein4,null,null,0,"List(26000055, 26000018, 26000024, 26000087, 26000012, 26000037, 28000001, 28000008)",2c49a230ab703adc4e5a1c859b10f081f316ce7e,#28CY90L9U,➹こうちゃん➷｜이렇게 짱,null,30,3,"List(26000011, 26000039, 27000012, 26000029, 28000007, 26000080, 26000025, 28000001)",eebc521e656df9115c6be54cc65fd78b84a8d663,false,#2899L9L,2026-06-02T15:16:16.576840+00:00,2026-06-03T16:04:50.797Z
00026e84cc8ccb2a829fd6818a52dbcd834f2938,2026-06-01T02:01:57+00:00,20260601T020157.000Z,pathOfLegend,false,Ranked1v1_NewArena2,54000145,Legendary Arena,#8QYLUL02Q,Y Eña,null,30,3,"List(26000063, 26000018, 26000004, 26000067, 28000012, 28000002, 28000001, 28000011)",41f2f16ce5855528f9657fba03b4dc89da3812cd,#2V2V20LQC,こたね,null,null,0,"List(27000002, 26000002, 26000056, 26000053, 26000005, 28000011, 28000009, 26000054)",92bbd7a528adb8bcacb92ad3314dda39814faf50,true,#8QYLUL02Q,2026-06-02T16:15:55.186533+00:00,2026-06-03T16:04:50.797Z
0002daf22f253a199ffe65a0e69c0e6a597e3267,2026-05-31T13:58:34+00:00,20260531T135834.000Z,riverRacePvP,false,CW_Battle_1v1,54000144,Spirit Square,#RPL92C0R,QLF59i,14000,null,1,"List(28000004, 26000039, 26000040, 26000083, 26000056, 26000041, 26000026, 27000004)",f69c3ea3c6ed54870646561792c98fa0f443db18,#8PYUCY8U,꧁✯Mason✯꧂,14000,null,3,"List(26000014, 26000003, 28000017, 28000010, 26000048, 26000034, 26000025, 28000001)",afbb7718b90130f98c25b7f7277d662167b6fb8c,false,#RPL92C0R,2026-06-02T16:34:46.719460+00:00,2026-06-03T16:04:50.797Z
0002f9c4377e3f01ecd36c6cca5c1854588cc305,2026-05-31T21:45:14+00:00,20260531T214514.000Z,pathOfLegend,false,Ranked1v1_NewArena2,54000145,Legendary Arena,#9RP00QGRC,Captain Kirk,1618,29,1,"List(26000056, 28000015, 27000002, 26000054, 28000000, 26000005, 26000053, 26000002)",fe972d582cc581e5e0d98b04b54bc7cbcd1d6f3a,#PGVP229LJ,Braian,1609,-29,0,"List(26000035, 26000034, 26000015, 26000037, 28000015, 26000006, 28000012, 28000005)",afbb14875d50381203df6e5b63ff6e1097d95a65,true,#9RP00QGRC,2026-06-02T16:15:31.023666+00:00,2026-06-03T16:04:50.797Z
00032a7475e600fa08c8101472a9c4e19afe3e4

## Bronze cards

Loads card dimension data into a schema-enforced Delta table (full overwrite).

In [0]:
cards_schema = T.StructType([
    T.StructField("card_id", T.LongType()),
    T.StructField("name", T.StringType()),
    T.StructField("rarity", T.StringType()),
    T.StructField("elixir_cost", T.LongType()),
    T.StructField("max_level", T.LongType()),
])
cards = (spark.read.schema(cards_schema).option("multiline", "true")
    .json(f"{RAW}/cards.json")
    .withColumn("bronze_loaded_at", F.current_timestamp()))
cards.write.format("delta").mode("overwrite").saveAsTable(BRONZE_CARDS)
print(spark.table(BRONZE_CARDS).count(), "cards")
display(spark.table(BRONZE_CARDS).orderBy("card_id").limit(10))

121 cards


card_id,name,rarity,elixir_cost,max_level,bronze_loaded_at
26000000,Knight,common,3,16,2026-06-03T18:14:36.483Z
26000001,Archers,common,3,16,2026-06-03T18:14:36.483Z
26000002,Goblins,common,2,16,2026-06-03T18:14:36.483Z
26000003,Giant,rare,5,14,2026-06-03T18:14:36.483Z
26000004,P.E.K.K.A,epic,7,11,2026-06-03T18:14:36.483Z
26000005,Minions,common,3,16,2026-06-03T18:14:36.483Z
26000006,Balloon,epic,5,11,2026-06-03T18:14:36.483Z
26000007,Witch,epic,5,11,2026-06-03T18:14:36.483Z
26000008,Barbarians,common,5,16,2026-06-03T18:14:36.483Z
26000009,Golem,epic,8,11,2026-06-03T18:14:36.483Z


## Bronze players (seed lineage)

Not an analytical table, just a record of the crawl seed for player battlelogs (overwrite)

In [0]:
players_schema = T.StructType([
    T.StructField("tag", T.StringType()),
    T.StructField("name", T.StringType()),
    T.StructField("trophies", T.LongType()),
    T.StructField("clan_tag", T.StringType()),
])
players = (spark.read.schema(players_schema).option("multiline", "true")
    .json(f"{RAW}/players.json")
    .withColumn("bronze_loaded_at", F.current_timestamp()))
players.write.format("delta").mode("overwrite").saveAsTable(BRONZE_PLAYERS)
print(spark.table(BRONZE_PLAYERS).count(), "players")
display(spark.table(BRONZE_PLAYERS).limit(10))

9964 players


tag,name,trophies,clan_tag,bronze_loaded_at
#2GURR8VLP,アスダシアス,14000,#QVL92090,2026-06-03T18:14:40.680Z
#92LRCVUY,Javav,14000,#QVL92090,2026-06-03T18:14:40.680Z
#PCPUU8Y8Y,Jerry,14000,#QVL92090,2026-06-03T18:14:40.680Z
#2YVJVQUJR,YodaBear,14000,#QVL92090,2026-06-03T18:14:40.680Z
#JP0G02,Jay,14000,#QVL92090,2026-06-03T18:14:40.680Z
#YLJL0JP,Lui,14000,#QVL92090,2026-06-03T18:14:40.680Z
#8LGVU09Y,Parsa,14000,#QVL92090,2026-06-03T18:14:40.680Z
#RY9J2VCPQ,Yae,14000,#QVL92090,2026-06-03T18:14:40.680Z
#GC2UJ0G,MacDonald81,14000,#QVL92090,2026-06-03T18:14:40.680Z
#8UPYRCYU,Combatiente,14000,#QVL92090,2026-06-03T18:14:40.680Z


## Done

Bronze is loaded and idempotent. Next: `02_silver_transform`.